In [1]:
import json
with open("./data/toolcall_3more_ocr_code_6200_cz_cleaned_1203.json") as f:
    datas = json.load(f)

In [3]:
len(datas)

6200

In [15]:
from typing import List, Dict, Any, Union, Optional

def check_role_alternation(messages: List[Dict[str, Any]]) -> Union[List[Dict[str, Any]], bool]:
    """
    检查消息列表中role是否按"user"-"assistant"交替出现
    
    Args:
        messages: 消息列表，每个消息是一个字典，包含'role'键
        
    Returns:
        如果符合规则，返回正确的消息列表（可能经过清理）
        如果不符合规则，返回False
    """
    if not messages:
        return messages  # 返回空列表
    
    def _check_sequence(msgs: List[Dict[str, Any]]) -> Optional[List[Dict[str, Any]]]:
        """内部函数：检查给定序列是否符合user-assistant交替规则"""
        if not msgs:
            return msgs
            
        for i, msg in enumerate(msgs):
            if 'role' not in msg:
                return None
                
            expected_role = 'user' if i % 2 == 0 else 'assistant'
            if msg['role'] != expected_role:
                return None
                
        return msgs
    
    # 首先检查原始序列
    result = _check_sequence(messages)
    if result is not None:
        return result
    
    # 如果原始序列不符合，检查是否有相邻的重复user记录
    # 查找是否有两个相邻的role为user的消息，且content相同
    for i in range(len(messages) - 1):
        current_msg = messages[i]
        next_msg = messages[i + 1]
        
        # 检查是否是相邻的两个user消息
        if (current_msg.get('role') == 'user' and 
            next_msg.get('role') == 'user' and
            'content' in current_msg and 
            'content' in next_msg and
            current_msg['content'] == next_msg['content']):
            
            # 创建一个新列表，删除其中一个重复的user消息
            # 方案1: 删除第二个重复的user消息
            cleaned_messages1 = messages.copy()
            cleaned_messages1.pop(i + 1)
            
            result1 = _check_sequence(cleaned_messages1)
            if result1 is not None:
                return result1
            
            # 方案2: 删除第一个重复的user消息
            cleaned_messages2 = messages.copy()
            cleaned_messages2.pop(i)
            
            result2 = _check_sequence(cleaned_messages2)
            if result2 is not None:
                return result2
    
    # 如果没有找到重复的user消息，但可能有其他修复方式
    # 例如：删除单个不匹配的消息，但注意这可能改变对话逻辑
    
    # 尝试删除第一个不符合预期的消息
    for i in range(len(messages)):
        expected_role = 'user' if i % 2 == 0 else 'assistant'
        if messages[i].get('role') != expected_role:
            # 尝试删除这个不符合预期的消息
            cleaned_messages = messages.copy()
            cleaned_messages.pop(i)
            
            result = _check_sequence(cleaned_messages)
            if result is not None:
                return result
    
    return False


In [ ]:
from rich import print as rprint

new_datas = []
for d in datas:
    if "Input Question" not in d['messages'][0]['content']:
        rprint(d)
        continue
        
    messages = check_role_alternation(d['messages'])
    if not messages:
        # rprint(d)
        print('='*10)
        continue
    
    for m in messages:
        if '\"VLSearchImage\"' in m['content']:
            m['content'] = m['content'].replace('\"VLSearchImage\"', '\"image_search\"')
    
    d.update({"messages": messages})
    new_datas.append(d)
    
print(len(new_datas))

In [21]:
with open("./data/toolcall_3more_ocr_code_6200_cz_cleaned_1216.jsonl", "w") as g:
    for d in new_datas:
        g.write(json.dumps(d, ensure_ascii=False)+'\n')
print(len(new_datas))

6196


In [24]:
with open('./data/toolcall_3more_ocr_code_6200_cz_cleaned_1216.jsonl') as f:
    for line in f:
        assert "VLSearchImage" not in line, rprint(line)